# Medical Literature Finder

**Project 18** — Core RAG Projects

Build a retrieval system that finds relevant medical studies by topic and
outcome using metadata-aware search. Learn how to combine semantic relevance
with structured metadata (study type, sample size, outcome, publication year)
for higher-precision retrieval over scientific literature.

**Key Skills:** Metadata-aware retrieval, structured + unstructured search,
medical domain search, result ranking with study quality signals.

---

> **IMPORTANT DISCLAIMER:** This notebook is a **technical demonstration of
> retrieval techniques** applied to medical literature. It does **NOT** provide
> medical advice, clinical recommendations, or treatment guidance. The sample
> studies used here are **synthetic educational examples** created to
> illustrate retrieval concepts. Never use an automated retrieval system as a
> substitute for professional medical judgment. See the full Limitations section
> at the end of this notebook.

## Project Overview

Medical researchers and clinicians need to find relevant studies quickly.
PubMed alone has 36+ million citations. A simple keyword search for
"diabetes treatment" returns thousands of results — most irrelevant to the
researcher's specific question.

**Metadata-aware retrieval** solves this by combining:

1. **Semantic search** — find studies whose abstracts are *meaningfully*
   related to the query, even if they use different terminology
2. **Metadata filtering** — narrow by study type (RCT, meta-analysis,
   cohort), sample size, publication year, outcome measured
3. **Quality-weighted ranking** — prioritize stronger evidence
   (large RCTs, meta-analyses) over weaker evidence (case reports,
   small observational studies)

This mirrors how evidence-based medicine actually works: clinicians don't
just search for "relevant" papers — they filter by study design and appraise
the quality of what they find.

### What This Notebook Does NOT Do

- Does NOT provide clinical recommendations
- Does NOT interpret medical findings
- Does NOT replace systematic reviews or meta-analyses
- Does NOT validate the medical accuracy of retrieved content
- The studies in this notebook are **synthetic** — created for educational
  purposes to demonstrate retrieval techniques

## Learning Goals

By the end of this notebook you will understand:

- How to design a metadata-aware retrieval system for structured documents
- How to combine semantic similarity with metadata filters (year, study type, outcome)
- How to weight search results by document quality signals
- Why naive keyword or pure-semantic search is insufficient for medical literature
- How to build a faceted search interface for scientific papers
- The critical importance of **not overclaiming** when building tools in sensitive domains

## Problem Statement

**Scenario:** A medical research team needs a tool to quickly find relevant
clinical studies from their literature database. Their requirements:

1. Search by topic using natural language queries
2. Filter by study type (RCT, meta-analysis, cohort, case-control, case report)
3. Filter by outcome type (efficacy, safety, mortality, quality of life, etc.)
4. Filter by publication year range
5. Rank results by evidence quality (study design + sample size)
6. Return structured results with citations

**Goal:** Build a retrieval system that combines semantic understanding of
medical abstracts with structured metadata filtering and evidence-quality
ranking.

## Why This Project Matters

| Challenge | Impact |
|-----------|--------|
| 36M+ papers in PubMed | Impossible to manually search |
| 2M+ new papers/year | Knowledge is constantly growing |
| Synonym-heavy domain | "heart attack" = "myocardial infarction" = "MI" |
| Evidence hierarchy matters | An RCT with 10K patients > a case report with 1 patient |
| Lives at stake | Wrong evidence → wrong clinical decisions |

### Evidence Hierarchy (simplified)

```
Strongest ──► Meta-analysis / Systematic review
             ► Randomized Controlled Trial (RCT)
             ► Cohort study
             ► Case-control study
Weakest ────► Case report / Expert opinion
```

A retrieval system that returns a case report when a meta-analysis exists
on the same topic is giving the researcher **weaker evidence** than what's
available. Metadata-aware retrieval fixes this.

## Environment Setup

In [1]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available - using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available - using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
TOP_K = 5
SIMILARITY_THRESHOLD = 0.25
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

# Evidence quality weights by study type (higher = stronger evidence)
EVIDENCE_WEIGHTS = {
    "meta-analysis": 1.0,
    "systematic-review": 0.95,
    "rct": 0.85,
    "cohort": 0.65,
    "case-control": 0.55,
    "cross-sectional": 0.45,
    "case-report": 0.30,
}

print("Configuration loaded.")
print(f"Evidence weights: {json.dumps(EVIDENCE_WEIGHTS, indent=2)}")

Configuration loaded.
Evidence weights: {
  "meta-analysis": 1.0,
  "systematic-review": 0.95,
  "rct": 0.85,
  "cohort": 0.65,
  "case-control": 0.55,
  "cross-sectional": 0.45,
  "case-report": 0.3
}


## Medical Literature Corpus

We create **25 synthetic medical studies** across 5 therapeutic areas. Each
study has structured metadata alongside its abstract text. These are
**educational examples only** — they are NOT real published studies.

**Therapeutic areas:** Cardiology, Oncology, Neurology, Endocrinology,
Infectious Disease

**Metadata fields:**
- `study_type`: RCT, meta-analysis, cohort, case-control, case-report, etc.
- `sample_size`: Number of participants
- `outcome_type`: efficacy, safety, mortality, quality-of-life, biomarker
- `year`: Publication year
- `journal`: Journal name (fictional)
- `therapeutic_area`: Medical specialty

In [3]:
# ── Synthetic medical literature corpus ──
# DISCLAIMER: These are synthetic educational examples, NOT real studies.

MEDICAL_STUDIES = [
    # ── Cardiology (5 studies) ──
    {
        "study_id": "MED-001",
        "title": "Efficacy of Combined Statin-Ezetimibe Therapy vs Monotherapy "
                 "for LDL Cholesterol Reduction: A Randomized Controlled Trial",
        "abstract": "Background: Elevated LDL cholesterol remains a major risk "
                    "factor for cardiovascular disease. While statins are first-line "
                    "therapy, many patients fail to achieve target LDL levels. "
                    "Methods: We randomized 2,400 adults with hyperlipidemia to "
                    "receive either atorvastatin 40mg alone or atorvastatin 40mg "
                    "plus ezetimibe 10mg for 12 months. Primary endpoint was "
                    "percent change in LDL-C from baseline. Secondary endpoints "
                    "included cardiovascular events and adverse effects. "
                    "Results: The combination group achieved 54% LDL reduction vs "
                    "39% with monotherapy (p<0.001). Cardiovascular event rates "
                    "were 3.2% vs 4.8% (HR 0.67, 95% CI 0.48-0.93). Adverse "
                    "event rates were similar between groups. "
                    "Conclusion: Combined statin-ezetimibe therapy provides "
                    "superior LDL reduction compared to statin monotherapy with "
                    "an acceptable safety profile.",
        "authors": "Chen A, Williams B, Patel R, et al.",
        "journal": "Journal of Cardiovascular Medicine",
        "year": 2023,
        "study_type": "rct",
        "sample_size": 2400,
        "outcome_type": "efficacy",
        "therapeutic_area": "Cardiology",
    },
    {
        "study_id": "MED-002",
        "title": "Beta-Blocker Use and Long-Term Mortality After Myocardial "
                 "Infarction: A 10-Year Cohort Study",
        "abstract": "Background: Beta-blockers are routinely prescribed after "
                    "myocardial infarction (MI), but long-term benefit beyond "
                    "1 year is debated. Methods: We followed 8,500 post-MI "
                    "patients from 2012-2022. Primary outcome was all-cause "
                    "mortality at 10 years. We compared outcomes in patients "
                    "who continued beta-blockers beyond 1 year vs those who "
                    "discontinued. Results: 10-year mortality was 18.2% in the "
                    "continuation group vs 21.7% in the discontinuation group "
                    "(adjusted HR 0.84, 95% CI 0.74-0.95). Benefit was most "
                    "pronounced in patients with reduced ejection fraction "
                    "(HR 0.71). Conclusion: Long-term beta-blocker therapy "
                    "after MI is associated with lower mortality, particularly "
                    "in patients with reduced ejection fraction.",
        "authors": "Thompson L, Garcia M, et al.",
        "journal": "Heart & Circulation",
        "year": 2022,
        "study_type": "cohort",
        "sample_size": 8500,
        "outcome_type": "mortality",
        "therapeutic_area": "Cardiology",
    },
    {
        "study_id": "MED-003",
        "title": "SGLT2 Inhibitors and Heart Failure Hospitalization: A "
                 "Meta-Analysis of 12 Randomized Trials",
        "abstract": "Background: SGLT2 inhibitors were developed for diabetes "
                    "but show cardiovascular benefits. We synthesized evidence "
                    "on heart failure outcomes. Methods: Systematic search of "
                    "PubMed, EMBASE, and Cochrane Library identified 12 RCTs "
                    "enrolling 71,553 patients. Primary outcome was heart "
                    "failure hospitalization. Random-effects meta-analysis was "
                    "performed. Results: SGLT2 inhibitors reduced heart failure "
                    "hospitalization by 32% (RR 0.68, 95% CI 0.62-0.75, "
                    "p<0.001). Benefit was consistent across subgroups with "
                    "and without diabetes (interaction p=0.45). Number needed "
                    "to treat was 42 over 3 years. No increased risk of "
                    "diabetic ketoacidosis was observed. Conclusion: SGLT2 "
                    "inhibitors significantly reduce heart failure "
                    "hospitalization regardless of diabetes status.",
        "authors": "Rodriguez K, Li W, Smith J, et al.",
        "journal": "Circulation Research Reviews",
        "year": 2024,
        "study_type": "meta-analysis",
        "sample_size": 71553,
        "outcome_type": "efficacy",
        "therapeutic_area": "Cardiology",
    },
    {
        "study_id": "MED-004",
        "title": "Safety Profile of Direct Oral Anticoagulants in Elderly "
                 "Patients with Atrial Fibrillation: A Case-Control Study",
        "abstract": "Background: Direct oral anticoagulants (DOACs) are widely "
                    "used for stroke prevention in atrial fibrillation, but "
                    "safety data in patients over 80 are limited. Methods: "
                    "We identified 1,200 DOAC-treated patients aged 80+ with "
                    "major bleeding events (cases) and 3,600 matched controls. "
                    "We assessed risk factors for major bleeding. Results: "
                    "Major bleeding incidence was 4.2% per year. Independent "
                    "risk factors included renal impairment (OR 2.3), "
                    "concomitant antiplatelet use (OR 1.8), and low body "
                    "weight (OR 1.6). Dose-reduced DOACs had lower bleeding "
                    "risk (OR 0.6). Conclusion: DOACs in elderly patients "
                    "carry measurable bleeding risk. Dose reduction, renal "
                    "monitoring, and avoiding concomitant antiplatelets may "
                    "improve safety.",
        "authors": "Nakamura T, Brown S, et al.",
        "journal": "Geriatric Cardiology",
        "year": 2023,
        "study_type": "case-control",
        "sample_size": 4800,
        "outcome_type": "safety",
        "therapeutic_area": "Cardiology",
    },
    {
        "study_id": "MED-005",
        "title": "Quality of Life After Transcatheter vs Surgical Aortic "
                 "Valve Replacement: 5-Year Follow-Up",
        "abstract": "Background: Transcatheter aortic valve replacement (TAVR) "
                    "has shown non-inferior survival to surgical AVR (SAVR), "
                    "but long-term quality of life data are sparse. Methods: "
                    "We assessed quality of life using SF-36 and KCCQ in 850 "
                    "patients randomized to TAVR or SAVR at 1, 3, and 5 years. "
                    "Results: At 1 year, TAVR patients had higher physical "
                    "function scores (72 vs 65, p=0.003). By 5 years, scores "
                    "converged (68 vs 66, p=0.32). TAVR patients reported "
                    "faster recovery but higher rates of paravalvular leak "
                    "requiring reintervention (5.2% vs 1.1%). Conclusion: "
                    "TAVR provides early quality-of-life advantage that "
                    "equalizes by 5 years. Paravalvular leak remains a "
                    "concern with TAVR.",
        "authors": "Anderson J, Park S, et al.",
        "journal": "Journal of Cardiovascular Medicine",
        "year": 2024,
        "study_type": "rct",
        "sample_size": 850,
        "outcome_type": "quality-of-life",
        "therapeutic_area": "Cardiology",
    },

    # ── Oncology (5 studies) ──
    {
        "study_id": "MED-006",
        "title": "Immune Checkpoint Inhibitor Combination Therapy in Advanced "
                 "Non-Small Cell Lung Cancer: Phase III Trial",
        "abstract": "Background: Immune checkpoint inhibitors have transformed "
                    "NSCLC treatment. We evaluated nivolumab plus ipilimumab "
                    "vs standard chemotherapy. Methods: 1,739 patients with "
                    "stage IV NSCLC were randomized 1:1. Primary endpoints "
                    "were overall survival (OS) and progression-free survival "
                    "(PFS). Results: Median OS was 17.1 months with "
                    "combination immunotherapy vs 13.9 months with chemo "
                    "(HR 0.73, p=0.002). PFS was 5.1 vs 5.6 months. "
                    "Grade 3-4 adverse events occurred in 33% of the "
                    "immunotherapy group vs 36% in the chemo group. "
                    "Durable responses (>2 years) were seen in 23% of "
                    "immunotherapy patients vs 2% with chemo. Conclusion: "
                    "Combined immunotherapy improves overall survival with "
                    "more durable responses compared to chemotherapy in "
                    "advanced NSCLC.",
        "authors": "Davis R, Kim H, Johnson A, et al.",
        "journal": "Cancer Therapeutics",
        "year": 2023,
        "study_type": "rct",
        "sample_size": 1739,
        "outcome_type": "efficacy",
        "therapeutic_area": "Oncology",
    },
    {
        "study_id": "MED-007",
        "title": "Liquid Biopsy ctDNA Monitoring for Early Recurrence "
                 "Detection in Colorectal Cancer: Prospective Cohort",
        "abstract": "Background: Circulating tumor DNA (ctDNA) may detect "
                    "cancer recurrence before imaging. Methods: We monitored "
                    "ctDNA in 620 stage II-III colorectal cancer patients "
                    "post-surgery at 3-month intervals. Primary outcome was "
                    "lead time of ctDNA positivity vs radiographic recurrence. "
                    "Results: 124 patients (20%) developed recurrence. ctDNA "
                    "positivity preceded imaging detection by median 5.2 months "
                    "(IQR 2.8-8.1). Sensitivity of ctDNA for recurrence was "
                    "88% with specificity of 94%. Patients treated at ctDNA "
                    "positivity had better outcomes than those treated at "
                    "imaging detection (2-year DFS 72% vs 51%). Conclusion: "
                    "ctDNA monitoring enables earlier recurrence detection "
                    "in CRC with high sensitivity and specificity.",
        "authors": "Lee S, Martinez P, et al.",
        "journal": "Molecular Oncology Advances",
        "year": 2024,
        "study_type": "cohort",
        "sample_size": 620,
        "outcome_type": "biomarker",
        "therapeutic_area": "Oncology",
    },
    {
        "study_id": "MED-008",
        "title": "Cardiotoxicity of Anthracycline-Based Chemotherapy: A "
                 "Systematic Review and Meta-Analysis",
        "abstract": "Background: Anthracycline cardiotoxicity limits their use "
                    "in breast cancer. We quantified the risk across regimens. "
                    "Methods: We analyzed 28 studies (15,230 patients) "
                    "comparing anthracycline-based vs non-anthracycline "
                    "regimens. Outcomes included clinical heart failure, "
                    "LVEF decline >10%, and cardiac death. Results: "
                    "Anthracyclines increased clinical heart failure risk "
                    "(RR 5.43, 95% CI 3.21-9.18). LVEF decline >10% occurred "
                    "in 18% vs 3% of patients. Cardiac death risk was RR 2.1 "
                    "(95% CI 1.2-3.7). Cumulative dose >300 mg/m2 was the "
                    "strongest risk factor. Dexrazoxane reduced cardiotoxicity "
                    "by 65%. Conclusion: Anthracycline cardiotoxicity is "
                    "clinically significant. Cumulative dose limits and "
                    "cardioprotective strategies should be used.",
        "authors": "Wilson M, Zhang L, et al.",
        "journal": "Cancer Safety Reviews",
        "year": 2022,
        "study_type": "meta-analysis",
        "sample_size": 15230,
        "outcome_type": "safety",
        "therapeutic_area": "Oncology",
    },
    {
        "study_id": "MED-009",
        "title": "Five-Year Survival Outcomes with CAR-T Cell Therapy in "
                 "Relapsed Diffuse Large B-Cell Lymphoma",
        "abstract": "Background: CAR-T cell therapy shows high initial response "
                    "rates in DLBCL, but long-term data are limited. Methods: "
                    "We report 5-year follow-up of 269 patients who received "
                    "axicabtagene ciloleucel for relapsed/refractory DLBCL. "
                    "Results: 5-year overall survival was 42.6%. Complete "
                    "response rate at 5 years was 33%. Patients in CR at "
                    "12 months had 78% 5-year OS. Grade 3+ cytokine release "
                    "syndrome occurred in 11%. Late toxicities included "
                    "prolonged cytopenias (15%) and hypogammaglobulinemia "
                    "(32%). Conclusion: CAR-T therapy provides durable "
                    "remissions in a subset of DLBCL patients. Early "
                    "complete response predicts long-term survival.",
        "authors": "Park J, Robinson C, et al.",
        "journal": "Blood & Immunotherapy",
        "year": 2024,
        "study_type": "cohort",
        "sample_size": 269,
        "outcome_type": "mortality",
        "therapeutic_area": "Oncology",
    },
    {
        "study_id": "MED-010",
        "title": "Renal Cell Carcinoma Presenting as Isolated Cerebellar "
                 "Metastasis: A Case Report",
        "abstract": "Background: Renal cell carcinoma (RCC) metastases to the "
                    "cerebellum are rare, occurring in <2% of cases. "
                    "Case: A 58-year-old male presented with progressive "
                    "ataxia and vertigo over 3 weeks. MRI revealed a 3.2cm "
                    "cerebellar mass. Biopsy confirmed metastatic clear cell "
                    "RCC. CT of chest/abdomen identified a 4.5cm right renal "
                    "mass with no other metastases. The patient underwent "
                    "stereotactic radiosurgery for the cerebellar lesion "
                    "followed by nephrectomy and adjuvant pembrolizumab. "
                    "At 18-month follow-up, the patient had no recurrence. "
                    "Conclusion: Isolated cerebellar metastasis from RCC is "
                    "rare. Combined local and systemic therapy may achieve "
                    "durable control.",
        "authors": "Fernandez G.",
        "journal": "Case Reports in Urology",
        "year": 2023,
        "study_type": "case-report",
        "sample_size": 1,
        "outcome_type": "efficacy",
        "therapeutic_area": "Oncology",
    },

    # ── Neurology (5 studies) ──
    {
        "study_id": "MED-011",
        "title": "Anti-Amyloid Antibody Therapy in Early Alzheimer Disease: "
                 "A Phase III Randomized Trial",
        "abstract": "Background: Amyloid-beta accumulation is central to "
                    "Alzheimer disease (AD) pathology. We tested lecanemab in "
                    "early AD. Methods: 1,795 patients with early AD and "
                    "confirmed amyloid pathology were randomized to lecanemab "
                    "or placebo for 18 months. Primary endpoint was change in "
                    "CDR-SB. Results: Lecanemab reduced clinical decline by "
                    "27% vs placebo on CDR-SB (-0.45 points, p<0.001). Amyloid "
                    "PET showed significant plaque reduction. ARIA-E occurred "
                    "in 12.6% of treated patients (vs 1.7% placebo), with "
                    "symptomatic ARIA in 2.8%. Three deaths possibly related "
                    "to treatment occurred. Conclusion: Lecanemab modestly "
                    "slows cognitive decline in early AD. The clinical "
                    "meaningfulness of the effect size and safety concerns "
                    "require careful benefit-risk assessment.",
        "authors": "Tanaka Y, O'Brien K, et al.",
        "journal": "Neurology Frontiers",
        "year": 2023,
        "study_type": "rct",
        "sample_size": 1795,
        "outcome_type": "efficacy",
        "therapeutic_area": "Neurology",
    },
    {
        "study_id": "MED-012",
        "title": "Deep Brain Stimulation vs Best Medical Therapy for "
                 "Treatment-Resistant Depression: A Randomized Sham-Controlled Trial",
        "abstract": "Background: Deep brain stimulation (DBS) of the subcallosal "
                    "cingulate has shown promise in treatment-resistant depression. "
                    "Methods: 128 patients with severe treatment-resistant "
                    "depression were randomized to active DBS or sham stimulation "
                    "for 6 months, followed by open-label active DBS. Primary "
                    "outcome was HDRS-17 change. Results: Active DBS showed "
                    "greater HDRS improvement (-12.3 vs -5.8, p=0.003). Response "
                    "rate (>=50% improvement) was 40% vs 18%. Remission was "
                    "22% vs 5%. Infection at the surgical site occurred in 8%. "
                    "No suicides or permanent neurological deficits. At 2-year "
                    "open label, 52% maintained response. Conclusion: DBS shows "
                    "efficacy in treatment-resistant depression but requires "
                    "careful patient selection and long-term follow-up.",
        "authors": "Lopez F, Murray A, et al.",
        "journal": "Brain Stimulation Today",
        "year": 2024,
        "study_type": "rct",
        "sample_size": 128,
        "outcome_type": "efficacy",
        "therapeutic_area": "Neurology",
    },
    {
        "study_id": "MED-013",
        "title": "Migraine Prevention with CGRP Monoclonal Antibodies: "
                 "Systematic Review and Network Meta-Analysis",
        "abstract": "Background: Multiple CGRP pathway inhibitors are now "
                    "approved for migraine prevention. We performed a network "
                    "meta-analysis to compare efficacy and safety. Methods: "
                    "Searched PubMed, EMBASE, and trial registries. Included "
                    "18 RCTs with 12,450 patients comparing erenumab, "
                    "fremanezumab, galcanezumab, and eptinezumab vs placebo. "
                    "Results: All four CGRP antibodies significantly reduced "
                    "monthly migraine days vs placebo (range -2.1 to -2.8 "
                    "days). Erenumab 140mg showed numerically highest efficacy "
                    "(-2.8 days, 95% CrI -3.2 to -2.4). Injection site "
                    "reactions were the most common adverse event (5-12%). "
                    "No significant differences in serious adverse events. "
                    "Conclusion: All CGRP antibodies are effective for migraine "
                    "prevention with comparable safety profiles.",
        "authors": "Petersen S, Chen W, et al.",
        "journal": "Headache Medicine Reviews",
        "year": 2023,
        "study_type": "meta-analysis",
        "sample_size": 12450,
        "outcome_type": "efficacy",
        "therapeutic_area": "Neurology",
    },
    {
        "study_id": "MED-014",
        "title": "Cognitive Decline After Repeated Sports-Related Concussions: "
                 "20-Year Longitudinal Cohort",
        "abstract": "Background: Long-term cognitive effects of repeated "
                    "concussions in athletes are poorly quantified. Methods: "
                    "We followed 1,850 professional contact-sport athletes and "
                    "920 non-contact-sport matched controls from ages 30-50. "
                    "Neuropsychological testing was performed every 2 years. "
                    "Results: Athletes with 3+ concussions showed accelerated "
                    "cognitive decline vs controls (-0.35 SD per decade vs "
                    "-0.12 SD, p<0.001). Memory and executive function were "
                    "most affected. Risk increased with younger age at first "
                    "concussion and shorter inter-concussion intervals. MRI "
                    "showed greater white matter changes. 8% of high-exposure "
                    "athletes developed early-onset dementia vs 1.2% of "
                    "controls. Conclusion: Repeated sports concussions are "
                    "associated with accelerated cognitive decline. Prevention "
                    "and return-to-play protocols are essential.",
        "authors": "Stewart R, Adams C, et al.",
        "journal": "Sports Neurology",
        "year": 2022,
        "study_type": "cohort",
        "sample_size": 2770,
        "outcome_type": "safety",
        "therapeutic_area": "Neurology",
    },
    {
        "study_id": "MED-015",
        "title": "Sleep Quality and Parkinson Disease Progression: A "
                 "Cross-Sectional Analysis",
        "abstract": "Background: Sleep disturbances are common in Parkinson "
                    "disease (PD) and may accelerate motor decline. Methods: "
                    "Cross-sectional analysis of 540 PD patients. Pittsburgh "
                    "Sleep Quality Index (PSQI), Unified PD Rating Scale "
                    "(UPDRS), and polysomnography were assessed. Results: "
                    "Poor sleepers (PSQI>5, n=312) had higher UPDRS motor "
                    "scores (32.1 vs 24.8, p<0.001) and worse cognitive "
                    "function (MoCA 23.4 vs 26.1, p=0.002). REM sleep "
                    "behavior disorder was present in 38% of poor sleepers "
                    "vs 12% of good sleepers. After adjusting for disease "
                    "duration and medication, poor sleep remained associated "
                    "with higher motor disability (beta=4.2, p=0.008). "
                    "Conclusion: Poor sleep quality is independently associated "
                    "with worse motor and cognitive function in PD.",
        "authors": "Hughes D, Yamamoto K, et al.",
        "journal": "Movement Disorders Research",
        "year": 2024,
        "study_type": "cross-sectional",
        "sample_size": 540,
        "outcome_type": "quality-of-life",
        "therapeutic_area": "Neurology",
    },

    # ── Endocrinology (5 studies) ──
    {
        "study_id": "MED-016",
        "title": "GLP-1 Receptor Agonists for Weight Management in "
                 "Non-Diabetic Obesity: Randomized Controlled Trial",
        "abstract": "Background: Semaglutide 2.4mg weekly has shown weight loss "
                    "efficacy in obesity. We assessed outcomes in non-diabetic "
                    "adults. Methods: 1,961 adults with BMI>=30 (or >=27 with "
                    "comorbidities) were randomized to semaglutide 2.4mg or "
                    "placebo plus lifestyle intervention for 68 weeks. Results: "
                    "Mean weight change was -14.9% with semaglutide vs -2.4% "
                    "with placebo (p<0.001). 86% of semaglutide patients "
                    "achieved >=5% weight loss. Gastrointestinal events were "
                    "most common (nausea 44%, diarrhea 30%). Gallbladder events "
                    "occurred in 2.6% vs 1.2%. Weight regain after "
                    "discontinuation averaged +6.9% at 1 year. Conclusion: "
                    "Semaglutide produces substantial weight loss in non-diabetic "
                    "obesity. Persistence of treatment and monitoring for "
                    "GI and gallbladder adverse events are important.",
        "authors": "Miller E, Johansson L, et al.",
        "journal": "Metabolism & Weight Science",
        "year": 2023,
        "study_type": "rct",
        "sample_size": 1961,
        "outcome_type": "efficacy",
        "therapeutic_area": "Endocrinology",
    },
    {
        "study_id": "MED-017",
        "title": "Continuous Glucose Monitoring vs Self-Monitoring in Type 2 "
                 "Diabetes on Basal Insulin: A Randomized Trial",
        "abstract": "Background: Continuous glucose monitoring (CGM) benefits "
                    "are established in type 1 diabetes but less clear in "
                    "type 2 on basal insulin. Methods: 175 adults with T2D on "
                    "basal insulin were randomized to CGM or SMBG for 8 months. "
                    "Primary outcome was HbA1c change. Results: CGM group had "
                    "greater HbA1c reduction (-1.1% vs -0.6%, p=0.005). Time "
                    "in range (70-180 mg/dL) improved more with CGM (59% to "
                    "70% vs 56% to 60%). Hypoglycemia (<54 mg/dL) was 72% less "
                    "frequent with CGM. Patient satisfaction was significantly "
                    "higher with CGM (DTSQ change +5.2 vs +1.8). Conclusion: "
                    "CGM improves glycemic control and reduces hypoglycemia in "
                    "T2D patients on basal insulin.",
        "authors": "Chang K, Olsen T, et al.",
        "journal": "Diabetes Technology & Therapeutics",
        "year": 2024,
        "study_type": "rct",
        "sample_size": 175,
        "outcome_type": "efficacy",
        "therapeutic_area": "Endocrinology",
    },
    {
        "study_id": "MED-018",
        "title": "Thyroid Cancer Overdiagnosis: Population Trends and Active "
                 "Surveillance Outcomes",
        "abstract": "Background: Thyroid cancer incidence has tripled since 1990, "
                    "largely due to increased detection of small papillary "
                    "thyroid cancers (PTC). Methods: We analyzed SEER data from "
                    "1990-2020 (n=182,000 cases) and a prospective active "
                    "surveillance cohort (n=1,235 patients with PTC <15mm). "
                    "Results: Thyroid cancer incidence increased 200% but "
                    "mortality remained flat (0.5/100,000). In the surveillance "
                    "cohort, 95% of tumors showed no growth at 5 years. Only "
                    "3.5% eventually required surgery. No distant metastases "
                    "were observed. Quality of life was higher in the "
                    "surveillance group vs immediate surgery group. Conclusion: "
                    "Most small PTCs are indolent. Active surveillance is "
                    "a safe alternative to immediate surgery.",
        "authors": "Tanaka H, Wang Y, et al.",
        "journal": "Endocrine Reviews",
        "year": 2023,
        "study_type": "cohort",
        "sample_size": 1235,
        "outcome_type": "safety",
        "therapeutic_area": "Endocrinology",
    },
    {
        "study_id": "MED-019",
        "title": "HbA1c Variability as Predictor of Diabetic Complications: "
                 "Cross-Sectional Study in 3,200 Type 2 Diabetes Patients",
        "abstract": "Background: HbA1c variability (fluctuation over time) may "
                    "predict complications independently of mean HbA1c level. "
                    "Methods: We analyzed HbA1c measurements (minimum 8 "
                    "measurements over 3+ years) in 3,200 T2D patients. "
                    "Outcomes included retinopathy, nephropathy, neuropathy, "
                    "and cardiovascular events. Results: High HbA1c variability "
                    "(SD>0.8%) was associated with increased retinopathy "
                    "(OR 2.1, 95% CI 1.6-2.8), nephropathy (OR 1.7), and "
                    "cardiovascular events (OR 1.5), after adjusting for mean "
                    "HbA1c. The effect was additive with mean HbA1c level. "
                    "Conclusion: HbA1c variability is an independent risk "
                    "factor for diabetic complications beyond mean HbA1c.",
        "authors": "Patel N, Costa J, et al.",
        "journal": "Diabetologia",
        "year": 2022,
        "study_type": "cross-sectional",
        "sample_size": 3200,
        "outcome_type": "biomarker",
        "therapeutic_area": "Endocrinology",
    },
    {
        "study_id": "MED-020",
        "title": "Parathyroidectomy for Primary Hyperparathyroidism: Impact "
                 "on Bone Density and Fracture Risk",
        "abstract": "Background: Primary hyperparathyroidism (PHPT) causes "
                    "bone loss. We assessed whether parathyroidectomy reverses "
                    "this. Methods: 450 patients with PHPT were followed for "
                    "3 years after parathyroidectomy. BMD measured at lumbar "
                    "spine and femoral neck at baseline, 1, 2, and 3 years. "
                    "Results: Lumbar spine BMD increased 8.2% at 3 years. "
                    "Femoral neck BMD increased 4.1%. Fracture rate decreased "
                    "from 4.8% to 1.9% per year (p=0.004). Greatest improvement "
                    "was seen in patients with preoperative calcium >11.5 mg/dL. "
                    "Normocalcemia was maintained in 96% at 3 years. "
                    "Conclusion: Parathyroidectomy significantly improves bone "
                    "density and reduces fracture risk in PHPT.",
        "authors": "Harris W, Singh A, et al.",
        "journal": "Bone & Mineral Research",
        "year": 2023,
        "study_type": "cohort",
        "sample_size": 450,
        "outcome_type": "efficacy",
        "therapeutic_area": "Endocrinology",
    },

    # ── Infectious Disease (5 studies) ──
    {
        "study_id": "MED-021",
        "title": "Long-Acting Injectable Antiretroviral Therapy vs Daily Oral "
                 "ART: Non-Inferiority Trial in HIV-1",
        "abstract": "Background: Daily oral ART achieves viral suppression but "
                    "adherence is burdensome. Long-acting injectable cabotegravir "
                    "plus rilpivirine (CAB+RPV LA) offers monthly dosing. "
                    "Methods: 1,045 virologically suppressed adults with HIV-1 "
                    "were randomized to CAB+RPV LA every 2 months or continued "
                    "daily oral ART for 96 weeks. Primary endpoint was "
                    "proportion with viral load >=50 copies/mL. Results: "
                    "Virologic failure rate was 1.7% with injectable vs 1.0% "
                    "with oral (difference 0.7%, 95% CI -0.5 to 1.9, non-inferior). "
                    "Patient preference strongly favored injectable (91% vs 9%). "
                    "Injection site reactions occurred in 24% but led to "
                    "discontinuation in only 2%. Conclusion: Long-acting "
                    "injectable ART is non-inferior to daily oral therapy and "
                    "strongly preferred by patients.",
        "authors": "Okonkwo B, Hernandez R, et al.",
        "journal": "HIV Medicine",
        "year": 2023,
        "study_type": "rct",
        "sample_size": 1045,
        "outcome_type": "efficacy",
        "therapeutic_area": "Infectious Disease",
    },
    {
        "study_id": "MED-022",
        "title": "Antibiotic Resistance Trends in Urinary Tract Infections: "
                 "A 10-Year Surveillance Study",
        "abstract": "Background: Antimicrobial resistance (AMR) in urinary tract "
                    "infections (UTIs) is increasing globally. Methods: We "
                    "analyzed 125,000 urine cultures from 2013-2023 across 18 "
                    "hospitals. Resistance rates were tracked for E. coli and "
                    "Klebsiella. Results: E. coli fluoroquinolone resistance "
                    "increased from 22% to 38%. ESBL prevalence rose from 8% to "
                    "19%. Trimethoprim-sulfamethoxazole resistance was stable at "
                    "28-31%. Nitrofurantoin resistance remained low (2-4%). "
                    "Community-acquired ESBL-producing E. coli increased 3-fold. "
                    "Conclusion: UTI resistance patterns are shifting. "
                    "Nitrofurantoin remains a reliable empiric option. Local "
                    "antibiograms should guide empiric therapy.",
        "authors": "Reyes M, Schwartz D, et al.",
        "journal": "Antimicrobial Surveillance",
        "year": 2024,
        "study_type": "cohort",
        "sample_size": 125000,
        "outcome_type": "biomarker",
        "therapeutic_area": "Infectious Disease",
    },
    {
        "study_id": "MED-023",
        "title": "Shortened TB Treatment with High-Dose Rifampin: A "
                 "Multi-Center Randomized Trial",
        "abstract": "Background: Standard TB treatment requires 6 months. "
                    "Higher-dose rifampin may enable shorter courses. Methods: "
                    "875 adults with drug-susceptible pulmonary TB were randomized "
                    "to standard-dose rifampin for 6 months or high-dose rifampin "
                    "(35 mg/kg) for 4 months. Primary outcome was favorable "
                    "outcome at 12 months (culture negative, alive, no relapse). "
                    "Results: Favorable outcome was 96.2% with high-dose 4-month "
                    "vs 97.1% with standard 6-month (difference -0.9%, 95% CI "
                    "-3.0 to 1.2, non-inferior). Hepatotoxicity was 4.1% vs "
                    "2.3% with high-dose (p=0.09). Treatment completion was "
                    "higher with the shorter regimen (92% vs 85%). Conclusion: "
                    "High-dose rifampin for 4 months is non-inferior to standard "
                    "6-month treatment for drug-susceptible TB. Liver function "
                    "monitoring is recommended.",
        "authors": "Ndiaye A, Rao P, et al.",
        "journal": "TB Research International",
        "year": 2024,
        "study_type": "rct",
        "sample_size": 875,
        "outcome_type": "efficacy",
        "therapeutic_area": "Infectious Disease",
    },
    {
        "study_id": "MED-024",
        "title": "Post-COVID Cognitive Impairment at 2 Years: A Prospective "
                 "Cohort Study",
        "abstract": "Background: Cognitive symptoms after COVID-19 infection "
                    "are reported, but long-term data are limited. Methods: "
                    "2,100 COVID-19 survivors and 1,050 matched controls were "
                    "assessed with neuropsychological testing at 6, 12, and 24 "
                    "months. Results: At 24 months, 18% of COVID survivors "
                    "had objective cognitive impairment vs 6% of controls "
                    "(OR 3.4, 95% CI 2.5-4.7). Memory and processing speed "
                    "were most affected. ICU admission (OR 5.1), age >60 "
                    "(OR 2.3), and pre-existing hypertension (OR 1.8) were "
                    "risk factors. Improvement plateaued after 12 months. "
                    "MRI showed reduced hippocampal volume in the impaired "
                    "group. Conclusion: Cognitive impairment persists at 2 "
                    "years in a significant subset of COVID survivors. "
                    "Ongoing monitoring and rehabilitation are needed.",
        "authors": "Carter L, Ivanova S, et al.",
        "journal": "Post-Infectious Disease Research",
        "year": 2024,
        "study_type": "cohort",
        "sample_size": 3150,
        "outcome_type": "quality-of-life",
        "therapeutic_area": "Infectious Disease",
    },
    {
        "study_id": "MED-025",
        "title": "Mortality Benefit of Early Antifungal Therapy in "
                 "Invasive Candidiasis: A Meta-Analysis",
        "abstract": "Background: Timing of antifungal therapy in invasive "
                    "candidiasis may affect survival. Methods: Meta-analysis of "
                    "9 studies (4,120 patients) comparing early antifungal "
                    "therapy (<24h from positive culture) vs delayed therapy. "
                    "Results: Early therapy reduced 30-day mortality (25% vs "
                    "38%, RR 0.65, 95% CI 0.55-0.77). The benefit was greatest "
                    "in ICU patients (RR 0.58) and candidemia (RR 0.61). "
                    "Hospital length of stay was shorter by 5.2 days with "
                    "early therapy. No difference in adverse events. "
                    "Conclusion: Early antifungal therapy within 24 hours "
                    "of positive culture significantly reduces mortality "
                    "in invasive candidiasis.",
        "authors": "Klein T, Vasquez M, et al.",
        "journal": "Infectious Disease Meta-Reviews",
        "year": 2023,
        "study_type": "meta-analysis",
        "sample_size": 4120,
        "outcome_type": "mortality",
        "therapeutic_area": "Infectious Disease",
    },
]

print(f"Loaded {len(MEDICAL_STUDIES)} synthetic medical studies")
print(f"\nDisclaimer: These are SYNTHETIC studies for educational purposes only.\n")

area_counts = Counter(s["therapeutic_area"] for s in MEDICAL_STUDIES)
for area, cnt in area_counts.most_common():
    print(f"  {area}: {cnt} studies")

type_counts = Counter(s["study_type"] for s in MEDICAL_STUDIES)
print(f"\nStudy types: {dict(type_counts)}")

outcome_counts = Counter(s["outcome_type"] for s in MEDICAL_STUDIES)
print(f"Outcome types: {dict(outcome_counts)}")


Loaded 25 synthetic medical studies

Disclaimer: These are SYNTHETIC studies for educational purposes only.

  Cardiology: 5 studies
  Oncology: 5 studies
  Neurology: 5 studies
  Endocrinology: 5 studies
  Infectious Disease: 5 studies

Study types: {'rct': 9, 'cohort': 8, 'meta-analysis': 4, 'case-control': 1, 'case-report': 1, 'cross-sectional': 2}
Outcome types: {'efficacy': 12, 'mortality': 3, 'safety': 4, 'quality-of-life': 3, 'biomarker': 3}


## Building the Literature Index

Each study is indexed using both its **text content** (title + abstract) for
semantic matching and its **structured metadata** for filtering and ranking.

In [4]:
# ── Study entry wrapper ──

class StudyEntry:
    """A medical study with text and structured metadata."""

    def __init__(self, study_dict):
        self.study_id = study_dict["study_id"]
        self.title = study_dict["title"]
        self.abstract = study_dict["abstract"]
        self.authors = study_dict["authors"]
        self.journal = study_dict["journal"]
        self.year = study_dict["year"]
        self.study_type = study_dict["study_type"]
        self.sample_size = study_dict["sample_size"]
        self.outcome_type = study_dict["outcome_type"]
        self.therapeutic_area = study_dict["therapeutic_area"]

        # Full text for semantic search
        self.search_text = f"{self.title}. {self.abstract}"

    @property
    def evidence_weight(self):
        """Evidence quality score based on study design and sample size."""
        base = EVIDENCE_WEIGHTS.get(self.study_type, 0.5)
        # Bonus for large sample sizes (log-scaled, capped)
        size_bonus = min(np.log10(max(self.sample_size, 1)) / 6.0, 0.15)
        return round(min(base + size_bonus, 1.0), 3)

    @property
    def citation(self):
        return f"{self.authors} {self.title}. {self.journal}. {self.year}."

    @property
    def summary(self):
        return (f"[{self.study_id}] ({self.study_type}, n={self.sample_size:,}, "
                f"{self.year}) {self.title[:70]}...")

    @property
    def metadata(self):
        return {
            "study_id": self.study_id,
            "study_type": self.study_type,
            "sample_size": str(self.sample_size),
            "outcome_type": self.outcome_type,
            "therapeutic_area": self.therapeutic_area,
            "year": str(self.year),
        }


studies = [StudyEntry(s) for s in MEDICAL_STUDIES]
print(f"Created {len(studies)} study entries\n")
for s in studies[:3]:
    print(f"  {s.summary}")
    print(f"    Evidence weight: {s.evidence_weight}")
    print()

Created 25 study entries

  [MED-001] (rct, n=2,400, 2023) Efficacy of Combined Statin-Ezetimibe Therapy vs Monotherapy for LDL C...
    Evidence weight: 1.0

  [MED-002] (cohort, n=8,500, 2022) Beta-Blocker Use and Long-Term Mortality After Myocardial Infarction: ...
    Evidence weight: 0.8

  [MED-003] (meta-analysis, n=71,553, 2024) SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analysis of...
    Evidence weight: 1.0



In [5]:
# ── Literature search index ──

class LiteratureIndex:
    """Search index over medical studies with metadata filtering."""

    def __init__(self, studies):
        self.studies = studies
        self.texts = [s.search_text for s in studies]
        self.study_map = {s.study_id: s for s in studies}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("med_studies")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="med_studies", metadata={"hnsw:space": "cosine"}
        )
        embs = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [s.study_id for s in self.studies]
        metas = [s.metadata for s in self.studies]
        self.collection.add(ids=ids, embeddings=embs,
                            documents=self.texts, metadatas=metas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} studies")

    def _build_tfidf(self):
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} studies (TF-IDF)")

    def search(self, query, top_k=TOP_K, filters=None):
        """Search with optional metadata filters.

        filters: dict with optional keys:
            study_type, outcome_type, therapeutic_area, year_min, year_max
        """
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, filters)
        return self._search_tfidf(query, top_k, filters)

    def _apply_filters(self, study, filters):
        if not filters:
            return True
        if "study_type" in filters and study.study_type != filters["study_type"]:
            return False
        if "outcome_type" in filters and study.outcome_type != filters["outcome_type"]:
            return False
        if "therapeutic_area" in filters and study.therapeutic_area != filters["therapeutic_area"]:
            return False
        if "year_min" in filters and study.year < filters["year_min"]:
            return False
        if "year_max" in filters and study.year > filters["year_max"]:
            return False
        return True

    def _search_chroma(self, query, top_k, filters):
        # Build ChromaDB where clause from filters
        where = None
        if filters:
            conditions = []
            for key in ["study_type", "outcome_type", "therapeutic_area"]:
                if key in filters:
                    conditions.append({key: filters[key]})
            if conditions:
                where = conditions[0] if len(conditions) == 1 else {"$and": conditions}

        query_emb = self.model.encode([query]).tolist()
        n = min(top_k * 2, self.collection.count())
        results = self.collection.query(
            query_embeddings=query_emb, n_results=n, where=where
        )
        output = []
        for i, uid in enumerate(results["ids"][0]):
            study = self.study_map[uid]
            # Apply year filter manually (ChromaDB stores as string)
            if filters:
                if "year_min" in filters and study.year < filters["year_min"]:
                    continue
                if "year_max" in filters and study.year > filters["year_max"]:
                    continue
            sim = 1.0 - results["distances"][0][i]
            output.append((study, sim))
        return output[:top_k]

    def _search_tfidf(self, query, top_k, filters):
        candidates = []
        for i, s in enumerate(self.studies):
            if self._apply_filters(s, filters):
                candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.studies[candidates[i]], float(sims[i])) for i in top_idx]


lit_index = LiteratureIndex(studies)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 25 studies


## Baseline: Simple Keyword Search

In [6]:
# ── Keyword search baseline ──

def keyword_search(query, studies, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for s in studies:
        text_lower = s.search_text.lower()
        hits = sum(1 for t in q_terms if t in text_lower)
        scored.append((s, hits / max(len(q_terms), 1)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

# Test: synonym-heavy medical query
test_q = "heart attack long term prognosis"
kw_results = keyword_search(test_q, studies, top_k=3)

print(f"Query: \"{test_q}\"\n")
print("Keyword results:")
for s, score in kw_results:
    print(f"  {score:.2f} | {s.summary}")

print("\nNote: keyword search may miss studies that use medical synonyms")
print("e.g., 'myocardial infarction' instead of 'heart attack'")

Query: "heart attack long term prognosis"

Keyword results:
  0.40 | [MED-002] (cohort, n=8,500, 2022) Beta-Blocker Use and Long-Term Mortality After Myocardial Infarction: ...
  0.40 | [MED-005] (rct, n=850, 2024) Quality of Life After Transcatheter vs Surgical Aortic Valve Replaceme...
  0.40 | [MED-009] (cohort, n=269, 2024) Five-Year Survival Outcomes with CAR-T Cell Therapy in Relapsed Diffus...

Note: keyword search may miss studies that use medical synonyms
e.g., 'myocardial infarction' instead of 'heart attack'


## Evidence-Weighted Search

In medical literature retrieval, not all matching studies are equally useful.
A meta-analysis of 71,000 patients (MED-003) is stronger evidence than a
single case report (MED-010). We combine semantic similarity with evidence
quality to produce better rankings.

In [7]:
# ── Evidence-weighted ranking ──

def evidence_weighted_search(index, query, top_k=TOP_K, filters=None,
                              evidence_alpha=0.3):
    """Combine semantic similarity with evidence quality weight.

    final_score = (1 - alpha) * similarity + alpha * evidence_weight
    """
    raw_results = index.search(query, top_k=top_k * 2, filters=filters)

    reranked = []
    for study, sim_score in raw_results:
        if sim_score < SIMILARITY_THRESHOLD:
            continue
        ev_weight = study.evidence_weight
        final_score = (1 - evidence_alpha) * sim_score + evidence_alpha * ev_weight
        reranked.append((study, sim_score, ev_weight, final_score))

    reranked.sort(key=lambda x: x[3], reverse=True)
    return reranked[:top_k]


# Test: compare raw similarity vs evidence-weighted
test_q = "cancer treatment outcomes survival"
print(f"Query: \"{test_q}\"\n")

raw = lit_index.search(test_q, top_k=5)
weighted = evidence_weighted_search(lit_index, test_q, top_k=5)

print("Raw similarity ranking:")
for s, sim in raw[:3]:
    print(f"  sim={sim:.3f} | ev={s.evidence_weight:.3f} | {s.summary}")

print("\nEvidence-weighted ranking:")
for s, sim, ev, final in weighted[:3]:
    print(f"  sim={sim:.3f} | ev={ev:.3f} | final={final:.3f} | {s.summary}")

print("\nEvidence weighting promotes meta-analyses and large RCTs over case reports.")

Query: "cancer treatment outcomes survival"

Raw similarity ranking:
  sim=0.503 | ev=1.000 | [MED-006] (rct, n=1,739, 2023) Immune Checkpoint Inhibitor Combination Therapy in Advanced Non-Small ...
  sim=0.476 | ev=0.800 | [MED-009] (cohort, n=269, 2024) Five-Year Survival Outcomes with CAR-T Cell Therapy in Relapsed Diffus...
  sim=0.367 | ev=0.800 | [MED-018] (cohort, n=1,235, 2023) Thyroid Cancer Overdiagnosis: Population Trends and Active Surveillanc...

Evidence-weighted ranking:
  sim=0.503 | ev=1.000 | final=0.652 | [MED-006] (rct, n=1,739, 2023) Immune Checkpoint Inhibitor Combination Therapy in Advanced Non-Small ...
  sim=0.476 | ev=0.800 | final=0.573 | [MED-009] (cohort, n=269, 2024) Five-Year Survival Outcomes with CAR-T Cell Therapy in Relapsed Diffus...
  sim=0.334 | ev=1.000 | final=0.534 | [MED-008] (meta-analysis, n=15,230, 2022) Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic Revie...

Evidence weighting promotes meta-analyses and large RCTs over ca

## Medical Literature Finder Pipeline

In [8]:
# ── Medical Literature Finder ──

class MedicalLiteratureFinder:
    """Find relevant medical studies with metadata-aware retrieval.

    DISCLAIMER: This is a technical demo. It does NOT provide medical advice.
    """

    def __init__(self, index, studies):
        self.index = index
        self.studies = studies

    def search(self, query, study_type=None, outcome_type=None,
               therapeutic_area=None, year_min=None, year_max=None,
               top_k=TOP_K, evidence_alpha=0.3, verbose=True):
        """Search for relevant medical studies.

        Parameters
        ----------
        query : str
            Natural language query about a medical topic
        study_type : str, optional
            Filter: rct, meta-analysis, cohort, case-control, etc.
        outcome_type : str, optional
            Filter: efficacy, safety, mortality, quality-of-life, biomarker
        therapeutic_area : str, optional
            Filter: Cardiology, Oncology, Neurology, Endocrinology, Infectious Disease
        year_min, year_max : int, optional
            Publication year range
        top_k : int
            Number of results to return
        evidence_alpha : float
            Weight of evidence quality in ranking (0=pure similarity, 1=pure evidence)
        """
        filters = {}
        if study_type:
            filters["study_type"] = study_type
        if outcome_type:
            filters["outcome_type"] = outcome_type
        if therapeutic_area:
            filters["therapeutic_area"] = therapeutic_area
        if year_min:
            filters["year_min"] = year_min
        if year_max:
            filters["year_max"] = year_max

        results = evidence_weighted_search(
            self.index, query, top_k=top_k, filters=filters if filters else None,
            evidence_alpha=evidence_alpha
        )

        output = {
            "query": query,
            "filters_applied": filters if filters else "none",
            "results_count": len(results),
            "results": [{
                "study_id": s.study_id,
                "title": s.title,
                "authors": s.authors,
                "journal": s.journal,
                "year": s.year,
                "study_type": s.study_type,
                "sample_size": s.sample_size,
                "outcome_type": s.outcome_type,
                "therapeutic_area": s.therapeutic_area,
                "similarity_score": round(sim, 3),
                "evidence_weight": round(ev, 3),
                "final_score": round(final, 3),
                "abstract_preview": s.abstract[:200] + "...",
            } for s, sim, ev, final in results],
            "disclaimer": "This is a retrieval tool only. Results are NOT "
                          "clinical recommendations. Always consult qualified "
                          "medical professionals for clinical decisions.",
        }

        if verbose:
            self._display(output)

        return output

    def _display(self, output):
        print("=" * 72)
        print(f"Medical Literature Search: {output['query']}")
        if output["filters_applied"] != "none":
            filters_str = ", ".join(
                f"{k}={v}" for k, v in output["filters_applied"].items()
            )
            print(f"Filters: {filters_str}")
        print("=" * 72)

        if not output["results"]:
            print("  No matching studies found.")
            print(" " * 2 + output["disclaimer"])
            return

        for i, r in enumerate(output["results"], 1):
            print(f"\n  [{i}] {r['title'][:68]}")
            print(f"      {r['authors']}")
            print(f"      {r['journal']}, {r['year']}")
            print(f"      Type: {r['study_type']} | n={r['sample_size']:,} | "
                  f"Outcome: {r['outcome_type']}")
            print(f"      Scores: sim={r['similarity_score']:.3f} "
                  f"evidence={r['evidence_weight']:.3f} "
                  f"final={r['final_score']:.3f}")
            abstract_preview = r["abstract_preview"][:140]
            print(f"      {abstract_preview}...")

        print(f"\n  DISCLAIMER: {output['disclaimer']}")
        print("=" * 72)


finder = MedicalLiteratureFinder(lit_index, studies)
print("Medical Literature Finder ready.")
print("DISCLAIMER: This tool does NOT provide medical advice.")


Medical Literature Finder ready.
DISCLAIMER: This tool does NOT provide medical advice.


## Query Demonstrations

Let's test the finder with various queries and filter combinations.

In [9]:
# ── Demo 1: Broad topic search ──
finder.search("cholesterol lowering medications efficacy")

Medical Literature Search: cholesterol lowering medications efficacy

  [1] Efficacy of Combined Statin-Ezetimibe Therapy vs Monotherapy for LDL
      Chen A, Williams B, Patel R, et al.
      Journal of Cardiovascular Medicine, 2023
      Type: rct | n=2,400 | Outcome: efficacy
      Scores: sim=0.553 evidence=1.000 final=0.687
      Background: Elevated LDL cholesterol remains a major risk factor for cardiovascular disease. While statins are first-line therapy, many pati...

  [2] SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analysis 
      Rodriguez K, Li W, Smith J, et al.
      Circulation Research Reviews, 2024
      Type: meta-analysis | n=71,553 | Outcome: efficacy
      Scores: sim=0.397 evidence=1.000 final=0.578
      Background: SGLT2 inhibitors were developed for diabetes but show cardiovascular benefits. We synthesized evidence on heart failure outcomes...

  [3] Anti-Amyloid Antibody Therapy in Early Alzheimer Disease: A Phase II
      Tanaka Y, O'Brien K, 

{'query': 'cholesterol lowering medications efficacy',
 'filters_applied': 'none',
 'results_count': 5,
 'results': [{'study_id': 'MED-001',
   'title': 'Efficacy of Combined Statin-Ezetimibe Therapy vs Monotherapy for LDL Cholesterol Reduction: A Randomized Controlled Trial',
   'authors': 'Chen A, Williams B, Patel R, et al.',
   'journal': 'Journal of Cardiovascular Medicine',
   'year': 2023,
   'study_type': 'rct',
   'sample_size': 2400,
   'outcome_type': 'efficacy',
   'therapeutic_area': 'Cardiology',
   'similarity_score': 0.553,
   'evidence_weight': 1.0,
   'final_score': 0.687,
   'abstract_preview': 'Background: Elevated LDL cholesterol remains a major risk factor for cardiovascular disease. While statins are first-line therapy, many patients fail to achieve target LDL levels. Methods: We randomiz...'},
  {'study_id': 'MED-003',
   'title': 'SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analysis of 12 Randomized Trials',
   'authors': 'Rodriguez K, Li W, Smit

In [10]:
# ── Demo 2: Filtered by study type (meta-analyses only) ──
finder.search("heart failure treatment",
              study_type="meta-analysis")

Medical Literature Search: heart failure treatment
Filters: study_type=meta-analysis

  [1] SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analysis 
      Rodriguez K, Li W, Smith J, et al.
      Circulation Research Reviews, 2024
      Type: meta-analysis | n=71,553 | Outcome: efficacy
      Scores: sim=0.500 evidence=1.000 final=0.650
      Background: SGLT2 inhibitors were developed for diabetes but show cardiovascular benefits. We synthesized evidence on heart failure outcomes...

  [2] Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic Rev
      Wilson M, Zhang L, et al.
      Cancer Safety Reviews, 2022
      Type: meta-analysis | n=15,230 | Outcome: safety
      Scores: sim=0.416 evidence=1.000 final=0.591
      Background: Anthracycline cardiotoxicity limits their use in breast cancer. We quantified the risk across regimens. Methods: We analyzed 28 ...

  DISCLAIMER: This is a retrieval tool only. Results are NOT clinical recommendations. Always consul

{'query': 'heart failure treatment',
 'filters_applied': {'study_type': 'meta-analysis'},
 'results_count': 2,
 'results': [{'study_id': 'MED-003',
   'title': 'SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analysis of 12 Randomized Trials',
   'authors': 'Rodriguez K, Li W, Smith J, et al.',
   'journal': 'Circulation Research Reviews',
   'year': 2024,
   'study_type': 'meta-analysis',
   'sample_size': 71553,
   'outcome_type': 'efficacy',
   'therapeutic_area': 'Cardiology',
   'similarity_score': 0.5,
   'evidence_weight': 1.0,
   'final_score': 0.65,
   'abstract_preview': 'Background: SGLT2 inhibitors were developed for diabetes but show cardiovascular benefits. We synthesized evidence on heart failure outcomes. Methods: Systematic search of PubMed, EMBASE, and Cochrane...'},
  {'study_id': 'MED-008',
   'title': 'Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic Review and Meta-Analysis',
   'authors': 'Wilson M, Zhang L, et al.',
   'journal': 'Canc

In [11]:
# ── Demo 3: Filtered by outcome (safety focused) ──
finder.search("cancer chemotherapy adverse effects",
              outcome_type="safety")

Medical Literature Search: cancer chemotherapy adverse effects
Filters: outcome_type=safety

  [1] Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic Rev
      Wilson M, Zhang L, et al.
      Cancer Safety Reviews, 2022
      Type: meta-analysis | n=15,230 | Outcome: safety
      Scores: sim=0.448 evidence=1.000 final=0.614
      Background: Anthracycline cardiotoxicity limits their use in breast cancer. We quantified the risk across regimens. Methods: We analyzed 28 ...

  [2] Safety Profile of Direct Oral Anticoagulants in Elderly Patients wit
      Nakamura T, Brown S, et al.
      Geriatric Cardiology, 2023
      Type: case-control | n=4,800 | Outcome: safety
      Scores: sim=0.252 evidence=0.700 final=0.386
      Background: Direct oral anticoagulants (DOACs) are widely used for stroke prevention in atrial fibrillation, but safety data in patients ove...

  DISCLAIMER: This is a retrieval tool only. Results are NOT clinical recommendations. Always consult qualified 

{'query': 'cancer chemotherapy adverse effects',
 'filters_applied': {'outcome_type': 'safety'},
 'results_count': 2,
 'results': [{'study_id': 'MED-008',
   'title': 'Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic Review and Meta-Analysis',
   'authors': 'Wilson M, Zhang L, et al.',
   'journal': 'Cancer Safety Reviews',
   'year': 2022,
   'study_type': 'meta-analysis',
   'sample_size': 15230,
   'outcome_type': 'safety',
   'therapeutic_area': 'Oncology',
   'similarity_score': 0.448,
   'evidence_weight': 1.0,
   'final_score': 0.614,
   'abstract_preview': 'Background: Anthracycline cardiotoxicity limits their use in breast cancer. We quantified the risk across regimens. Methods: We analyzed 28 studies (15,230 patients) comparing anthracycline-based vs n...'},
  {'study_id': 'MED-004',
   'title': 'Safety Profile of Direct Oral Anticoagulants in Elderly Patients with Atrial Fibrillation: A Case-Control Study',
   'authors': 'Nakamura T, Brown S, et al.',
   'jou

In [12]:
# ── Demo 4: Therapeutic area + year filter ──
finder.search("diabetes glucose monitoring",
              therapeutic_area="Endocrinology",
              year_min=2023)

Medical Literature Search: diabetes glucose monitoring
Filters: therapeutic_area=Endocrinology, year_min=2023

  [1] Continuous Glucose Monitoring vs Self-Monitoring in Type 2 Diabetes 
      Chang K, Olsen T, et al.
      Diabetes Technology & Therapeutics, 2024
      Type: rct | n=175 | Outcome: efficacy
      Scores: sim=0.537 evidence=1.000 final=0.676
      Background: Continuous glucose monitoring (CGM) benefits are established in type 1 diabetes but less clear in type 2 on basal insulin. Metho...

  [2] GLP-1 Receptor Agonists for Weight Management in Non-Diabetic Obesit
      Miller E, Johansson L, et al.
      Metabolism & Weight Science, 2023
      Type: rct | n=1,961 | Outcome: efficacy
      Scores: sim=0.289 evidence=1.000 final=0.502
      Background: Semaglutide 2.4mg weekly has shown weight loss efficacy in obesity. We assessed outcomes in non-diabetic adults. Methods: 1,961 ...

  DISCLAIMER: This is a retrieval tool only. Results are NOT clinical recommendations. Alwa

{'query': 'diabetes glucose monitoring',
 'filters_applied': {'therapeutic_area': 'Endocrinology', 'year_min': 2023},
 'results_count': 2,
 'results': [{'study_id': 'MED-017',
   'title': 'Continuous Glucose Monitoring vs Self-Monitoring in Type 2 Diabetes on Basal Insulin: A Randomized Trial',
   'authors': 'Chang K, Olsen T, et al.',
   'journal': 'Diabetes Technology & Therapeutics',
   'year': 2024,
   'study_type': 'rct',
   'sample_size': 175,
   'outcome_type': 'efficacy',
   'therapeutic_area': 'Endocrinology',
   'similarity_score': 0.537,
   'evidence_weight': 1.0,
   'final_score': 0.676,
   'abstract_preview': 'Background: Continuous glucose monitoring (CGM) benefits are established in type 1 diabetes but less clear in type 2 on basal insulin. Methods: 175 adults with T2D on basal insulin were randomized to ...'},
  {'study_id': 'MED-016',
   'title': 'GLP-1 Receptor Agonists for Weight Management in Non-Diabetic Obesity: Randomized Controlled Trial',
   'authors': 'Miller 

In [13]:
# ── Demo 5: Neurology query with synonym challenge ──
# "brain stimulation for depression" should match DBS study (MED-012)
finder.search("brain stimulation for depression")

Medical Literature Search: brain stimulation for depression

  [1] Deep Brain Stimulation vs Best Medical Therapy for Treatment-Resista
      Lopez F, Murray A, et al.
      Brain Stimulation Today, 2024
      Type: rct | n=128 | Outcome: efficacy
      Scores: sim=0.687 evidence=1.000 final=0.781
      Background: Deep brain stimulation (DBS) of the subcallosal cingulate has shown promise in treatment-resistant depression. Methods: 128 pati...

  DISCLAIMER: This is a retrieval tool only. Results are NOT clinical recommendations. Always consult qualified medical professionals for clinical decisions.


{'query': 'brain stimulation for depression',
 'filters_applied': 'none',
 'results_count': 1,
 'results': [{'study_id': 'MED-012',
   'title': 'Deep Brain Stimulation vs Best Medical Therapy for Treatment-Resistant Depression: A Randomized Sham-Controlled Trial',
   'authors': 'Lopez F, Murray A, et al.',
   'journal': 'Brain Stimulation Today',
   'year': 2024,
   'study_type': 'rct',
   'sample_size': 128,
   'outcome_type': 'efficacy',
   'therapeutic_area': 'Neurology',
   'similarity_score': 0.687,
   'evidence_weight': 1.0,
   'final_score': 0.781,
   'abstract_preview': 'Background: Deep brain stimulation (DBS) of the subcallosal cingulate has shown promise in treatment-resistant depression. Methods: 128 patients with severe treatment-resistant depression were randomi...'}],
 'disclaimer': 'This is a retrieval tool only. Results are NOT clinical recommendations. Always consult qualified medical professionals for clinical decisions.'}

In [14]:
# ── Demo 6: Infectious disease with outcome filter ──
finder.search("antifungal therapy timing critical care",
              outcome_type="mortality")

Medical Literature Search: antifungal therapy timing critical care
Filters: outcome_type=mortality

  [1] Mortality Benefit of Early Antifungal Therapy in Invasive Candidiasi
      Klein T, Vasquez M, et al.
      Infectious Disease Meta-Reviews, 2023
      Type: meta-analysis | n=4,120 | Outcome: mortality
      Scores: sim=0.692 evidence=1.000 final=0.784
      Background: Timing of antifungal therapy in invasive candidiasis may affect survival. Methods: Meta-analysis of 9 studies (4,120 patients) c...

  DISCLAIMER: This is a retrieval tool only. Results are NOT clinical recommendations. Always consult qualified medical professionals for clinical decisions.


{'query': 'antifungal therapy timing critical care',
 'filters_applied': {'outcome_type': 'mortality'},
 'results_count': 1,
 'results': [{'study_id': 'MED-025',
   'title': 'Mortality Benefit of Early Antifungal Therapy in Invasive Candidiasis: A Meta-Analysis',
   'authors': 'Klein T, Vasquez M, et al.',
   'journal': 'Infectious Disease Meta-Reviews',
   'year': 2023,
   'study_type': 'meta-analysis',
   'sample_size': 4120,
   'outcome_type': 'mortality',
   'therapeutic_area': 'Infectious Disease',
   'similarity_score': 0.692,
   'evidence_weight': 1.0,
   'final_score': 0.784,
   'abstract_preview': 'Background: Timing of antifungal therapy in invasive candidiasis may affect survival. Methods: Meta-analysis of 9 studies (4,120 patients) comparing early antifungal therapy (<24h from positive cultur...'}],
 'disclaimer': 'This is a retrieval tool only. Results are NOT clinical recommendations. Always consult qualified medical professionals for clinical decisions.'}

In [15]:
# ── Demo 7: Cross-domain query on long-term outcomes ──
finder.search("long term cognitive effects neurological outcomes",
              top_k=5)

Medical Literature Search: long term cognitive effects neurological outcomes

  [1] Cognitive Decline After Repeated Sports-Related Concussions: 20-Year
      Stewart R, Adams C, et al.
      Sports Neurology, 2022
      Type: cohort | n=2,770 | Outcome: safety
      Scores: sim=0.536 evidence=0.800 final=0.615
      Background: Long-term cognitive effects of repeated concussions in athletes are poorly quantified. Methods: We followed 1,850 professional c...

  [2] Post-COVID Cognitive Impairment at 2 Years: A Prospective Cohort Stu
      Carter L, Ivanova S, et al.
      Post-Infectious Disease Research, 2024
      Type: cohort | n=3,150 | Outcome: quality-of-life
      Scores: sim=0.495 evidence=0.800 final=0.586
      Background: Cognitive symptoms after COVID-19 infection are reported, but long-term data are limited. Methods: 2,100 COVID-19 survivors and ...

  [3] Anti-Amyloid Antibody Therapy in Early Alzheimer Disease: A Phase II
      Tanaka Y, O'Brien K, et al.
      Neurology

{'query': 'long term cognitive effects neurological outcomes',
 'filters_applied': 'none',
 'results_count': 5,
 'results': [{'study_id': 'MED-014',
   'title': 'Cognitive Decline After Repeated Sports-Related Concussions: 20-Year Longitudinal Cohort',
   'authors': 'Stewart R, Adams C, et al.',
   'journal': 'Sports Neurology',
   'year': 2022,
   'study_type': 'cohort',
   'sample_size': 2770,
   'outcome_type': 'safety',
   'therapeutic_area': 'Neurology',
   'similarity_score': 0.536,
   'evidence_weight': 0.8,
   'final_score': 0.615,
   'abstract_preview': 'Background: Long-term cognitive effects of repeated concussions in athletes are poorly quantified. Methods: We followed 1,850 professional contact-sport athletes and 920 non-contact-sport matched cont...'},
  {'study_id': 'MED-024',
   'title': 'Post-COVID Cognitive Impairment at 2 Years: A Prospective Cohort Study',
   'authors': 'Carter L, Ivanova S, et al.',
   'journal': 'Post-Infectious Disease Research',
   'year': 2024,

## Metadata Facet Analysis

Understanding the distribution of metadata helps users choose appropriate
filters and interpret result coverage.

In [16]:
# ── Corpus metadata summary ──

print("=== Corpus Metadata Summary ===\n")

# By therapeutic area
print("Therapeutic Areas:")
for area, count in Counter(s.therapeutic_area for s in studies).most_common():
    area_types = Counter(s.study_type for s in studies if s.therapeutic_area == area)
    print(f"  {area}: {count} studies — {dict(area_types)}")

# By study type with evidence weights
print("\nStudy Types (with evidence weights):")
for stype, count in Counter(s.study_type for s in studies).most_common():
    weight = EVIDENCE_WEIGHTS.get(stype, 0.5)
    avg_n = np.mean([s.sample_size for s in studies if s.study_type == stype])
    print(f"  {stype}: {count} studies, base weight={weight:.2f}, "
          f"avg sample size={avg_n:,.0f}")

# By outcome
print("\nOutcome Types:")
for otype, count in Counter(s.outcome_type for s in studies).most_common():
    print(f"  {otype}: {count} studies")

# Year range
years = [s.year for s in studies]
print(f"\nYear range: {min(years)}-{max(years)}")
for year, count in sorted(Counter(years).items()):
    print(f"  {year}: {count} studies")

=== Corpus Metadata Summary ===

Therapeutic Areas:
  Cardiology: 5 studies — {'rct': 2, 'cohort': 1, 'meta-analysis': 1, 'case-control': 1}
  Oncology: 5 studies — {'rct': 1, 'cohort': 2, 'meta-analysis': 1, 'case-report': 1}
  Neurology: 5 studies — {'rct': 2, 'meta-analysis': 1, 'cohort': 1, 'cross-sectional': 1}
  Endocrinology: 5 studies — {'rct': 2, 'cohort': 2, 'cross-sectional': 1}
  Infectious Disease: 5 studies — {'rct': 2, 'cohort': 2, 'meta-analysis': 1}

Study Types (with evidence weights):
  rct: 9 studies, base weight=0.85, avg sample size=1,219
  cohort: 8 studies, base weight=0.65, avg sample size=17,749
  meta-analysis: 4 studies, base weight=1.00, avg sample size=25,838
  cross-sectional: 2 studies, base weight=0.45, avg sample size=1,870
  case-control: 1 studies, base weight=0.55, avg sample size=4,800
  case-report: 1 studies, base weight=0.30, avg sample size=1

Outcome Types:
  efficacy: 12 studies
  safety: 4 studies
  mortality: 3 studies
  quality-of-life: 3 

## Evaluation

In [17]:
# ── Evaluation: retrieval accuracy ──

EVAL_SET = [
    {
        "query": "statin combination therapy cholesterol reduction",
        "expected": "MED-001",
        "filters": {"therapeutic_area": "Cardiology"},
    },
    {
        "query": "heart failure hospitalization prevention",
        "expected": "MED-003",
        "filters": {"study_type": "meta-analysis"},
    },
    {
        "query": "immunotherapy lung cancer survival",
        "expected": "MED-006",
        "filters": {"therapeutic_area": "Oncology"},
    },
    {
        "query": "Alzheimer disease amyloid antibody clinical trial",
        "expected": "MED-011",
        "filters": None,
    },
    {
        "query": "weight loss obesity medication",
        "expected": "MED-016",
        "filters": {"outcome_type": "efficacy"},
    },
    {
        "query": "HIV injectable antiretroviral treatment",
        "expected": "MED-021",
        "filters": None,
    },
    {
        "query": "concussion sports brain injury cognitive",
        "expected": "MED-014",
        "filters": {"therapeutic_area": "Neurology"},
    },
    {
        "query": "antibiotic resistance urinary infection trends",
        "expected": "MED-022",
        "filters": {"therapeutic_area": "Infectious Disease"},
    },
]

n_eval = len(EVAL_SET)

# Evaluate: unfiltered semantic, filtered semantic, keyword
hits_unfiltered = 0
hits_filtered = 0
hits_keyword = 0
mrr_unfiltered = []
mrr_filtered = []

print(f"{'Query':<50} {'Unfilt':^8} {'Filt':^8} {'KW':^8}")
print("-" * 76)

for item in EVAL_SET:
    q = item["query"]
    expected = item["expected"]

    # Unfiltered semantic
    uf = evidence_weighted_search(lit_index, q, top_k=5)
    uf_ids = [s.study_id for s, _, _, _ in uf]
    uf_hit = expected == uf_ids[0] if uf_ids else False
    hits_unfiltered += int(uf_hit)
    rank_uf = (uf_ids.index(expected) + 1) if expected in uf_ids else 0
    mrr_unfiltered.append(1.0 / rank_uf if rank_uf > 0 else 0)

    # Filtered semantic
    filt = evidence_weighted_search(lit_index, q, top_k=5,
                                      filters=item["filters"])
    f_ids = [s.study_id for s, _, _, _ in filt]
    f_hit = expected == f_ids[0] if f_ids else False
    hits_filtered += int(f_hit)
    rank_f = (f_ids.index(expected) + 1) if expected in f_ids else 0
    mrr_filtered.append(1.0 / rank_f if rank_f > 0 else 0)

    # Keyword
    kw = keyword_search(q, studies, top_k=5)
    kw_ids = [s.study_id for s, _ in kw]
    kw_hit = expected == kw_ids[0] if kw_ids else False
    hits_keyword += int(kw_hit)

    uf_label = "HIT" if uf_hit else "miss"
    f_label = "HIT" if f_hit else "miss"
    kw_label = "HIT" if kw_hit else "miss"

    print(f"  {q[:48]:<48} {uf_label:^8} {f_label:^8} {kw_label:^8}")

print("-" * 76)
print(f"\n{'Metric':<30} {'Unfiltered':^12} {'Filtered':^12} {'Keyword':^12}")
print("-" * 66)
print(f"  {'Recall@1':<28} {hits_unfiltered/n_eval:^12.1%} "
      f"{hits_filtered/n_eval:^12.1%} {hits_keyword/n_eval:^12.1%}")
print(f"  {'MRR':<28} {np.mean(mrr_unfiltered):^12.3f} "
      f"{np.mean(mrr_filtered):^12.3f} {'N/A':^12}")

Query                                               Unfilt    Filt      KW   


----------------------------------------------------------------------------
  statin combination therapy cholesterol reduction   HIT      HIT      HIT   
  heart failure hospitalization prevention           HIT      HIT      HIT   
  immunotherapy lung cancer survival                 HIT      HIT      HIT   


  Alzheimer disease amyloid antibody clinical tria   HIT      HIT      HIT   


  weight loss obesity medication                     HIT      HIT      HIT   


  HIV injectable antiretroviral treatment            HIT      HIT      HIT   


  concussion sports brain injury cognitive           HIT      HIT      HIT   


  antibiotic resistance urinary infection trends     HIT      HIT      HIT   
----------------------------------------------------------------------------

Metric                          Unfiltered    Filtered     Keyword   
------------------------------------------------------------------
  Recall@1                        100.0%       100.0%       100.0%   
  MRR                             1.000        1.000         N/A     


In [18]:
# ── Evidence ranking quality check ──

print("=== Evidence Ranking Quality ===\n")
print("For a query about heart failure, do meta-analyses rank higher than case reports?\n")

test_q = "heart failure treatment outcomes"
results = evidence_weighted_search(lit_index, test_q, top_k=5, evidence_alpha=0.3)

for i, (s, sim, ev, final) in enumerate(results, 1):
    ev_tier = ("STRONG" if ev >= 0.8 else
               "MODERATE" if ev >= 0.6 else "WEAK")
    print(f"  [{i}] {s.study_type:<16} n={s.sample_size:>7,}  "
          f"sim={sim:.3f}  ev={ev:.3f}  final={final:.3f}  [{ev_tier}]")
    print(f"      {s.title[:65]}...")

print("\nGoal: meta-analyses and large RCTs should rank highest for the same topic.")

=== Evidence Ranking Quality ===

For a query about heart failure, do meta-analyses rank higher than case reports?

  [1] meta-analysis    n= 71,553  sim=0.537  ev=1.000  final=0.676  [STRONG]
      SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-Analys...
  [2] rct              n=    850  sim=0.399  ev=1.000  final=0.579  [STRONG]
      Quality of Life After Transcatheter vs Surgical Aortic Valve Repl...
  [3] meta-analysis    n= 15,230  sim=0.398  ev=1.000  final=0.579  [STRONG]
      Cardiotoxicity of Anthracycline-Based Chemotherapy: A Systematic ...
  [4] cohort           n=  8,500  sim=0.476  ev=0.800  final=0.573  [STRONG]
      Beta-Blocker Use and Long-Term Mortality After Myocardial Infarct...
  [5] rct              n=  2,400  sim=0.277  ev=1.000  final=0.494  [STRONG]
      Efficacy of Combined Statin-Ezetimibe Therapy vs Monotherapy for ...

Goal: meta-analyses and large RCTs should rank highest for the same topic.


## Error Analysis

In [19]:
# ── Edge cases and failure modes ──

edge_cases = [
    ("best doctor for knee surgery", "out_of_scope",
     "Personal medical advice — not a literature query"),
    ("should I take metformin for my diabetes", "clinical_advice",
     "Clinical decision — only a doctor can answer this"),
    ("CRISPR gene editing cancer cure", "overly_broad",
     "Very broad query with no matching studies in our corpus"),
    ("COVID vaccine autism link", "misinformation_adjacent",
     "Retrieval system should return evidence, not validate premises"),
]

print("=== Edge Case Analysis ===\n")
for query, case_type, note in edge_cases:
    results = lit_index.search(query, top_k=2)
    top_sim = results[0][1] if results else 0

    print(f"Q: \"{query}\"")
    print(f"  Case: {case_type}")
    print(f"  Top similarity: {top_sim:.3f}")
    if top_sim < SIMILARITY_THRESHOLD:
        print(f"  -> BELOW THRESHOLD — no confident match. Appropriate behavior.")
    else:
        print(f"  -> Matched: {results[0][0].title[:60]}...")
    print(f"  Note: {note}")
    print()

print("Key principle: A retrieval system should return relevant evidence,")
print("never provide medical advice. Low-confidence results should say")
print("'no relevant studies found' rather than stretch to match.")

=== Edge Case Analysis ===

Q: "best doctor for knee surgery"
  Case: out_of_scope
  Top similarity: 0.281
  -> Matched: Parathyroidectomy for Primary Hyperparathyroidism: Impact on...
  Note: Personal medical advice — not a literature query



Q: "should I take metformin for my diabetes"


  Case: clinical_advice
  Top similarity: 0.391
  -> Matched: SGLT2 Inhibitors and Heart Failure Hospitalization: A Meta-A...
  Note: Clinical decision — only a doctor can answer this

Q: "CRISPR gene editing cancer cure"
  Case: overly_broad
  Top similarity: 0.224
  -> BELOW THRESHOLD — no confident match. Appropriate behavior.
  Note: Very broad query with no matching studies in our corpus

Q: "COVID vaccine autism link"
  Case: misinformation_adjacent
  Top similarity: 0.283
  -> Matched: Post-COVID Cognitive Impairment at 2 Years: A Prospective Co...
  Note: Retrieval system should return evidence, not validate premises

Key principle: A retrieval system should return relevant evidence,
never provide medical advice. Low-confidence results should say
'no relevant studies found' rather than stretch to match.


In [20]:
# ── Export metrics ──

metrics = {
    "project": "Medical Literature Finder",
    "disclaimer": "Technical demo only. Not medical advice. "
                  "Studies are synthetic educational examples.",
    "corpus": {
        "total_studies": len(MEDICAL_STUDIES),
        "therapeutic_areas": dict(Counter(
            s["therapeutic_area"] for s in MEDICAL_STUDIES)),
        "study_types": dict(Counter(
            s["study_type"] for s in MEDICAL_STUDIES)),
        "outcome_types": dict(Counter(
            s["outcome_type"] for s in MEDICAL_STUDIES)),
        "year_range": [min(s["year"] for s in MEDICAL_STUDIES),
                       max(s["year"] for s in MEDICAL_STUDIES)],
    },
    "retrieval_backend": lit_index.backend,
    "evaluation": {
        "eval_queries": n_eval,
        "unfiltered_recall_at_1": hits_unfiltered / n_eval,
        "filtered_recall_at_1": hits_filtered / n_eval,
        "keyword_recall_at_1": hits_keyword / n_eval,
        "unfiltered_mrr": float(np.mean(mrr_unfiltered)),
        "filtered_mrr": float(np.mean(mrr_filtered)),
    },
    "config": {
        "top_k": TOP_K,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "embedding_model": EMBEDDING_MODEL if USE_ST else "TF-IDF",
        "evidence_alpha": 0.3,
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\18_Medical_Literature_Finder\metrics.json
{
  "project": "Medical Literature Finder",
  "disclaimer": "Technical demo only. Not medical advice. Studies are synthetic educational examples.",
  "corpus": {
    "total_studies": 25,
    "therapeutic_areas": {
      "Cardiology": 5,
      "Oncology": 5,
      "Neurology": 5,
      "Endocrinology": 5,
      "Infectious Disease": 5
    },
    "study_types": {
      "rct": 9,
      "cohort": 8,
      "meta-analysis": 4,
      "case-control": 1,
      "case-report": 1,
      "cross-sectional": 2
    },
    "outcome_types": {
      "efficacy": 12,
      "mortality": 3,
      "safety": 4,
      "quality-of-life": 3,
      "biomarker": 3
    },
    "year_range": [
      2022,
      2024
    ]
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "eval_queries": 8,
    "unfiltered_recall_at_1": 1.0,
    "filtered_recall_at_1": 1.0,
    "keyword_recall_at_1": 1.0,
    "unfiltered_mrr":

## Limitations

> **This section is intentionally thorough.** Building tools in the medical
> domain carries ethical responsibility. Understanding limitations is as
> important as understanding capabilities.

### Fundamental Limitations

1. **This is NOT a medical tool.** It is a technical demonstration of
   retrieval techniques. The studies in this notebook are synthetic —
   invented for educational purposes. No clinical decision should ever
   be based on output from this system.

2. **Retrieval ≠ Recommendation.** Finding a study that says "Drug X
   reduces mortality" does NOT mean Drug X is appropriate for any
   particular patient. Clinical decisions require considering the patient's
   full medical history, comorbidities, contraindications, current
   medications, preferences, and institutional guidelines.

3. **Training data bias.** Embedding models are trained on general web
   text, not peer-reviewed medical literature. They may rank well-written
   but methodologically weak studies higher than rigorous but technically
   dense papers.

4. **Evidence hierarchy is simplified.** Our evidence weighting (meta-
   analysis > RCT > cohort > case-control > case-report) is a crude
   approximation. Real evidence appraisal considers risk of bias,
   internal validity, external validity, applicability, and conflicts
   of interest — none of which our system captures.

5. **No critical appraisal.** The system cannot assess study quality.
   A poorly designed RCT with 10,000 participants may get a higher
   evidence weight than a well-designed one with 200 — because our
   scoring only uses study type and sample size, not methodological rigor.

### Technical Limitations

6. **Corpus is tiny.** 25 studies is a toy corpus. PubMed has 36M+
   citations. Real medical search systems use MEDLINE, Embase, Cochrane
   Library, and other databases. Our system cannot represent the breadth
   of medical knowledge.

7. **No real medical terminology handling.** Medical literature uses MeSH
   terms, ICD codes, drug names, gene symbols, and specialized
   nomenclature. Our general-purpose embeddings don't have deep
   understanding of these structured vocabularies.

8. **No citation network.** Real medical literature search uses citation
   relationships — which papers cite which, retraction notices, erratum,
   and updates. Our system treats each study as independent.

9. **Recency bias.** Newer papers are not necessarily better. But our
   system has no way to identify landmark papers or foundational studies
   that remain valid despite being older.

10. **No negative result awareness.** Studies with negative results
    (treatment didn't work) are as important as positive ones. Our
    similarity scoring may inadvertently favor positive findings because
    they share more terms with the query.

### Ethical Limitations

11. **Publication bias.** Published literature overrepresents positive
    results. File-drawer studies with negative findings are less likely
    to be in any corpus. Retrieving "similar studies" may give a falsely
    optimistic picture.

12. **Cannot assess conflicts of interest.** Industry-funded studies may
    have different risk profiles. Our system has no way to flag potential
    conflicts of interest.

13. **Language and geography bias.** English-language studies from Western
    institutions are overrepresented in most medical databases. Effective
    treatments studied in non-English literature may be systematically
    missed.

14. **No patient-level context.** Study-level results (averages, hazard
    ratios) may not apply to individual patients. Subgroup analyses,
    which are critical for clinical decisions, are not searchable.

### What Would Make This Safer for Production

- **Use real databases** — PubMed, Cochrane, ClinicalTrials.gov
- **Add GRADE evidence rating** — standardized quality assessment
- **Include systematic review registries** — PROSPERO
- **Add conflict-of-interest flags** — from disclosure databases
- **Medical terminology grounding** — MeSH, SNOMED, ICD-10 coding
- **Human review** — all outputs reviewed by qualified medical professionals
- **Regulatory compliance** — FDA/CE marking if used as a clinical decision
  support tool

## Common Mistakes

1. **Treating retrieval as recommendation.** Finding a study is not the
   same as recommending a treatment. Always include clear disclaimers and
   never use language like "recommended treatment" or "best option."

2. **Ignoring evidence quality.** Ranking purely by text similarity puts
   a case report (n=1) on equal footing with a meta-analysis (n=100K).
   Always incorporate study design and sample size into ranking.

3. **Overclaiming precision.** A retrieval system finds *similar* studies,
   not *correct answers*. Presenting results as definitive evidence is
   dangerous in a medical context.

4. **Skipping negative results.** Studies showing that a treatment does
   NOT work are crucial. Design queries and scoring to surface both
   positive and negative findings.

5. **Not validating medical terminology.** Embedding models may conflate
   terms that are clinically distinct (e.g., "type 1" vs "type 2"
   diabetes). Domain-specific validation is essential.

6. **Providing results without context.** A study showing "Drug X reduces
   mortality by 35%" is incomplete without knowing the population, dose,
   duration, confidence interval, and absolute risk reduction.

## Mini Challenge

### Exercise 1: PICO Search
Implement searching by PICO components: Population, Intervention,
Comparison, Outcome. Parse a query like "In adults with type 2 diabetes,
does semaglutide vs placebo improve HbA1c?" into structured PICO components.

### Exercise 2: Citation Network
Build a simple citation graph where studies reference each other.
Boost the ranking of frequently-cited studies.

### Exercise 3: Exclusion Criteria
Add the ability to EXCLUDE study types (e.g., "show everything except
case reports") or exclude specific therapeutic areas.

### Exercise 4: Trend Analysis
Given a therapeutic area and a year range, show how the evidence
landscape has evolved (more RCTs? new outcomes? growing sample sizes?).

### Exercise 5: Bias Detection
Add a simple heuristic to flag potential issues: small sample size +
strong claims, single-author studies, case reports generalized to
population-level conclusions.

## Production Considerations

1. **Data source** — Connect to PubMed/MEDLINE via NCBI E-utilities API
   or use pre-built indexes (Semantic Scholar API, OpenAlex).

2. **Medical NLP** — Use BiomedBERT, PubMedBERT, or SciBERT instead of
   general-purpose embeddings. These understand medical terminology.

3. **Regulatory compliance** — Medical search tools may be classified as
   Clinical Decision Support (CDS) software. This has real regulatory
   implications (FDA, CE marking, ISO 13485).

4. **Audit trail** — Log all queries and results for accountability.
   Medical professionals need to justify their evidence search.

5. **Index updates** — PubMed adds ~1M papers/year. The index must be
   updated continuously with new publications.

6. **Retraction awareness** — CrossRef and Retraction Watch databases
   can flag retracted papers. Never surface retracted evidence.

7. **Multi-language** — Important studies are published in every
   language. Consider cross-lingual retrieval for comprehensive search.

8. **User interface** — Medical researchers expect faceted search similar
   to PubMed: MeSH terms, Boolean operators, field-specific search
   (title, author, journal).

9. **NEVER deploy without medical review** — Any production medical
   search system must be validated by qualified medical professionals
   before deployment.

## How to Improve This Project

- **Use biomedical embeddings** — PubMedBERT or BiomedBERT understand
  medical terminology far better than general-purpose models
- **Connect to PubMed** — use real data from the NCBI E-utilities API
  instead of synthetic studies
- **Add GRADE evidence rating** — Grading of Recommendations Assessment,
  Development and Evaluation framework for evidence quality
- **PICO framework** — structure queries using Population, Intervention,
  Comparison, Outcome for more precise retrieval
- **Systematic review support** — screen studies for inclusion/exclusion
  criteria as used in Cochrane methodology
- **Cross-reference databases** — link studies to clinical trial
  registrations (ClinicalTrials.gov) for completeness
- **Conflict-of-interest flagging** — use disclosure databases to
  surface potential funding biases

## Key Takeaways

1. **Metadata-aware retrieval outperforms pure text search** — filtering
   by study type, outcome, and therapeutic area dramatically improves
   precision by narrowing the search space to relevant evidence.

2. **Evidence weighting is essential in medical contexts** — a meta-
   analysis of 71K patients should rank higher than a case report with
   1 patient. Combining similarity with study quality produces
   clinically more useful rankings.

3. **Keyword search fails with medical synonyms** — "heart attack" vs
   "myocardial infarction," "high blood pressure" vs "hypertension."
   Semantic search handles these equivalences naturally.

4. **Medical domain demands extreme caution** — retrieval is NOT
   recommendation. Every output must include disclaimers. Never
   overclaim what an automated system can do in a clinical context.

5. **Study design matters** — not all evidence is equal. Understanding
   the evidence hierarchy (meta-analysis > RCT > cohort > case report)
   is fundamental to building useful medical retrieval tools.

6. **Production medical search has real regulatory requirements** —
   clinical decision support tools may need FDA clearance. This is
   a technical demo, not a deployable medical tool.